In [ ]:
# --- Ensure working directory is project root (contains 'parameter/') ---
import os
if not os.path.isdir('parameter') and os.path.isdir('../parameter'):
    os.chdir('..')


In [ ]:
# --- Pick per-case data via CASE_ID env var ---
import os
CASE_ID = os.environ.get('CASE_ID')
if CASE_ID is None:
    raise RuntimeError("CASE_ID env var must be set (e.g. 'case0_N5').")
print(f'Running case: {CASE_ID}')


In [ ]:
import numpy as np
from numpy.linalg import norm
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import cvxpy as cp
import daqp
from matplotlib.patches import Polygon, Circle
import pickle
import time

# ==============================================================================
# MA-CBF-VO (Multi-Agent CBF-VO) baseline implementation
# - Hard CBF: HOCBF on h1 = ||p_ij||^2 - critical^2  (ho_cbf from ho_cbf_part, unchanged)
# - Soft VO : h_vo (vo_cbf from vo_cbf_part, unchanged) + slack lambda
# - Objective: 0.5 p_u ||w - w_nominal||^2  +  k_vo * sum(1/T_col_i) * lambda_i^2
# - Dynamics: 5D const-speed (input = heading rate omega)
# ==============================================================================


In [ ]:
# ==============================================================================
# 1. System dynamics definition (multi-agent)
# ==============================================================================

def multi_agent_const_speed_5d_dynamics(t, y, u_all_flat, n_agents, state_dim, V_const):
    """
    Dynamics of N_AGENTS 5D constant-speed models.
    State: [px, py, vx, vy, theta], control: [omega]
    """
    current_states = y.reshape((state_dim, n_agents), order='F')
    control_inputs = u_all_flat.reshape((1, n_agents), order='F')

    vx = current_states[2, :]
    vy = current_states[3, :]
    theta = current_states[4, :]
    omega = control_inputs[0, :]

    d_state = np.zeros_like(current_states)
    d_state[0, :] = vx
    d_state[1, :] = vy
    d_state[2, :] = -V_const * np.sin(theta) * omega
    d_state[3, :] =  V_const * np.cos(theta) * omega
    d_state[4, :] = omega

    return d_state.flatten('F')

In [ ]:
def relative_state(STATE_DIM, N_AGENTS, current_states_matrix):
    """
    Relative state between every pair of agents (vectorized).
    """
    p = current_states_matrix[0:2, :]
    v = current_states_matrix[2:4, :]
    seta = current_states_matrix[4, :]

    relative_p_tensor = p[:, :, np.newaxis] - p[:, np.newaxis, :]
    relative_v_tensor = v[:, :, np.newaxis] - v[:, np.newaxis, :]

    norm_relative_p = np.linalg.norm(relative_p_tensor, axis=0)
    norm_relative_v = np.linalg.norm(relative_v_tensor, axis=0)
    dot_relative_pv = np.sum(relative_p_tensor * relative_v_tensor, axis=0)
    relative_seta = np.sin(seta[np.newaxis, :] - seta[:, np.newaxis])

    vector_seta_all = np.vstack([-np.sin(seta), np.cos(seta)])
    dot_relative_p_seta = np.sum(vector_seta_all[:, :, np.newaxis] * (-relative_p_tensor), axis=0)

    return norm_relative_p, norm_relative_v, dot_relative_pv, relative_seta, dot_relative_p_seta

In [ ]:
def ACS_flocking(u_limit, beta, lamda, k1, k2, desired_distance, N_AGENTS, V_CONST,
                 pij_norm, vij_norm, pv_norm, seta_ij, p_dot_seta):
    """
    ACS_flocking nominal controller (vectorized).
    """
    align_weight_matrix = (1 + (pij_norm)**2)**(-beta)
    u_align_all = (lamda / N_AGENTS) * np.sum(align_weight_matrix * seta_ij, axis=1)

    pij_norm_safe = pij_norm.copy()
    np.fill_diagonal(pij_norm_safe, 1.0)

    term_A_matrix = (k1 / (2 * pij_norm_safe**2)) * pv_norm
    term_B_matrix = (k2 * (pij_norm_safe - desired_distance) / (2 * pij_norm_safe)) * p_dot_seta
    combined_matrix = term_A_matrix + term_B_matrix
    np.fill_diagonal(combined_matrix, 0)

    u_inter_all = (1 / (N_AGENTS * V_CONST)) * np.sum(combined_matrix, axis=1)

    flocking_control = u_align_all + u_inter_all
    flocking_control = np.clip(flocking_control, -u_limit, u_limit)
    flocking_control = np.trunc(flocking_control * 1000) / 1000

    return flocking_control.reshape(-1, 1)

In [ ]:
def minimum_inter_distance_function(pij_norm, N_AGENTS):
    pij_with_inf_diag = pij_norm.copy()
    np.fill_diagonal(pij_with_inf_diag, np.inf)
    return np.min(pij_with_inf_diag)

In [ ]:
def maximum_inter_distance_function(pij_norm, N_AGENTS):
    pij_with_inf_diag = pij_norm.copy()
    np.fill_diagonal(pij_with_inf_diag, -1*np.inf)
    return np.max(pij_with_inf_diag)

In [ ]:
def ho_cbf(pij_norm, vij_norm, pv_norm, N_AGENTS, V_CONST, critical_distance,
           current_states_matrix, class_k1, class_k2, margin):
    """
    Hard safety CBF: HOCBF (Higher-Order CBF) on h1 = ||p_ij||^2 - critical^2.
    Used in place of h_scbf (braking-distance form) from the original MA-CBF-VO.

    Returns:
      h1         : (M,1) raw CBF value
      constraint : (M,1) Lf^2 h + class_k1*Lfh + class_k2*(Lfh + class_k1*h) - margin
                   (i.e. inequality Lgh * w + constraint >= 0)
      Lgh        : (M, N) Lie derivative w.r.t. input omega
    """
    np.seterr(divide='warn', invalid='warn')


    matrix_size = (N_AGENTS * (N_AGENTS - 1)) // 2
    rows, cols = np.triu_indices(N_AGENTS, k=1)

    seta = current_states_matrix[4, :]
    p_vector = current_states_matrix[0:2, :]
    v_vector = current_states_matrix[2:4, :]

    pij_vectors = p_vector[:, rows] - p_vector[:, cols]
    vij_vectors = v_vector[:, rows] - v_vector[:, cols]

    pij_norm_pairs = pij_norm[rows, cols]
    vij_norm_pairs = vij_norm[rows, cols]
    pv_norm_pairs  = pv_norm[rows, cols]

    h1 = (pij_norm_pairs**2 - critical_distance**2)

    grad_vij = 2 * pij_vectors

    lf2 = 2 * (vij_norm_pairs**2)
    caculate1 = class_k1 * (2 * pv_norm_pairs)
    caculate2 = class_k2 * (2 * pv_norm_pairs + class_k1 * (h1))
    psi1 = 2 * pv_norm_pairs + class_k1 * h1   # psi_1 = Lfh + class_k1*h  (first-order HOCBF invariant)
    constraint = lf2 + caculate1 + caculate2   # h2 (canonical HO-CBF, no margin)

    g_i_all = V_CONST * np.vstack([-np.sin(seta), np.cos(seta)])
    g_j_all = V_CONST * np.vstack([ np.sin(seta), -np.cos(seta)])

    g_i_pairs = g_i_all[:, rows]
    g_j_pairs = g_j_all[:, cols]

    lg_i = np.sum(g_i_pairs * grad_vij, axis=0)
    lg_j = np.sum(g_j_pairs * grad_vij, axis=0)

    Lgh = np.zeros((matrix_size, N_AGENTS))
    row_indices_for_lgh = np.arange(matrix_size)
    Lgh[row_indices_for_lgh, rows] = lg_i
    Lgh[row_indices_for_lgh, cols] = lg_j

    return h1.reshape(-1, 1), psi1.reshape(-1, 1), constraint.reshape(-1, 1), Lgh

In [ ]:
def vo_cbf(pij_norm, vij_norm, pv_norm, N_AGENTS, V_CONST, critical_distance, current_states_matrix, margin):
    """
    Soft VO CBF: h_vo = p_AB . v_AB + ||v_AB|| * sqrt(||p_AB||^2 - R^2)
                       (= p_AB . v_AB + ||v_AB|| * d_AB * cos(gamma)).
    Equivalent to h_vo in the MA-CBF-VO paper.

    Returns:
      h_x : (M,1)
      Lfx : (M,1)   (drift term)
      Lgh : (M, N)  (Lie derivative w.r.t. input omega)
    Inequality:  Lgh * w + Lfx + alpha_vo * h_x  >=  -lambda_i
    """
    np.seterr(divide='warn', invalid='warn')


    matrix_size = (N_AGENTS * (N_AGENTS - 1)) // 2
    rows, cols = np.triu_indices(N_AGENTS, k=1)

    seta = current_states_matrix[4, :]
    p_vector = current_states_matrix[0:2, :]
    v_vector = current_states_matrix[2:4, :]

    pij_vectors = p_vector[:, rows] - p_vector[:, cols]
    vij_vectors = v_vector[:, rows] - v_vector[:, cols]

    pij_norm_pairs = pij_norm[rows, cols]
    vij_norm_pairs = vij_norm[rows, cols]
    pv_norm_pairs  = pv_norm[rows, cols]

    h_x = np.zeros(matrix_size)
    Lfx = np.zeros(matrix_size)
    grad_vij = np.zeros((2, matrix_size))

    sqrt_term_all = np.sqrt(pij_norm_pairs**2 - critical_distance**2)

    grad_term = (sqrt_term_all / vij_norm_pairs)
    grad_vij = pij_vectors + grad_term[np.newaxis, :] * vij_vectors

    h_x = pv_norm_pairs + vij_norm_pairs * sqrt_term_all

    ca1 = vij_norm_pairs / sqrt_term_all
    Lfx = (vij_norm_pairs**2) + ca1 * pv_norm_pairs


    g_i_all = V_CONST * np.vstack([-np.sin(seta), np.cos(seta)])
    g_j_all = V_CONST * np.vstack([ np.sin(seta), -np.cos(seta)])

    g_i_pairs = g_i_all[:, rows]
    g_j_pairs = g_j_all[:, cols]

    lg_i = np.sum(g_i_pairs * grad_vij, axis=0)
    lg_j = np.sum(g_j_pairs * grad_vij, axis=0)

    Lgh = np.zeros((matrix_size, N_AGENTS))
    row_indices_for_lgh = np.arange(matrix_size)
    Lgh[row_indices_for_lgh, rows] = lg_i
    Lgh[row_indices_for_lgh, cols] = lg_j

    return h_x.reshape(-1, 1), Lfx.reshape(-1, 1), Lgh

In [ ]:
def time_to_collision_pairs(pij_vectors, vij_vectors, critical_distance):
    """
    Time-to-collision for each agent pair (i,j) under the constant-speed assumption.
    Solve  ||p_ij + t v_ij||^2 = R^2  for smallest positive t.

    Args:
        pij_vectors: (2, M) relative position vectors (p_i - p_j)
        vij_vectors: (2, M) relative velocity vectors (v_i - v_j)
        critical_distance: safety radius R = R_A + R_B

    Returns:
        t_col: (M,) time-to-collision. inf if no collision, small positive if already overlapping.
    """
    a = np.sum(vij_vectors**2, axis=0)
    b = 2.0 * np.sum(pij_vectors * vij_vectors, axis=0)
    c = np.sum(pij_vectors**2, axis=0) - critical_distance**2

    M = a.shape[0]
    t_col = np.full(M, np.inf)

    a_safe = np.where(a > 1e-12, a, 1e-12)
    disc = b**2 - 4.0 * a_safe * c
    sqrt_disc = np.sqrt(np.maximum(disc, 0.0))
    t1 = (-b - sqrt_disc) / (2.0 * a_safe)

    # Standard case where a collision is projected in the future
    mask_future = (disc >= 0) & (a > 1e-12) & (t1 > 0) & (c > 0)
    t_col[mask_future] = t1[mask_future]

    # Already overlapping (rarely happens numerically, but kept as a safeguard)
    overlap = c <= 0
    t_col[overlap] = 1e-3

    return t_col

In [ ]:
def solve_ma_cbf_vo_qp_with_warm_start(N, M, n_var, n_in,
                                       ho_constraint, lgh_ho,
                                       h_vo, lfh_vo, lgh_vo,
                                       T_col, k_vo, alpha_vo,
                                       u_nominal, u_limit, t,
                                       qp_max_iter, qp_eps_abs, qp_time_limit):
    """
    MA-CBF-VO QP via DAQP directly.
    Decision variables: x = [omega(N); lambda(M)]
    Returns (omega_vec, reason) where reason is one of:
        'ok', 'qp_nan', 'qp_infeasible', 'qp_iter_limit', 'qp_time_limit', 'qp_fail_<flag>'.
    """
    global _last_daqp_solve_time
    p_u = 0.5

    H = np.zeros((n_var, n_var), dtype=np.float64)
    H[:N, :N] = p_u * np.eye(N)
    T_safe = np.where(np.isfinite(T_col) & (T_col > 1e-6), T_col, 1e6)
    H[N:, N:] = np.diag(2.0 * k_vo / T_safe)

    f = np.zeros(n_var, dtype=np.float64)
    f[:N] = -p_u * u_nominal.flatten()

    # Inequality rows (upper-bounded by h_qp, lower-bounded by -inf):
    # 1) Hard HOCBF: -Lgh_ho * omega <= ho_constraint
    # 2) Soft VO:    -Lgh_vo * omega - lambda <= lfh_vo + alpha_vo * h_vo
    G_ho = np.hstack([-lgh_ho.astype(np.float64), np.zeros((M, M))])
    G_vo = np.hstack([-lgh_vo.astype(np.float64), -np.eye(M)])
    h_ho = ho_constraint.flatten().astype(np.float64)
    h_vo_rhs = (lfh_vo.flatten() + alpha_vo * h_vo.flatten()).astype(np.float64)

    G_qp = np.vstack([G_ho, G_vo])
    h_qp = np.concatenate([h_ho, h_vo_rhs])

    if np.any(np.isnan(G_qp)) or np.any(np.isnan(h_qp)) or np.any(np.isnan(f)):
        print(f"COLLISION (QP input NaN/Inf) at t={t}")
        _last_daqp_solve_time = 0.0; return None, 'collision'

    # Simple bounds: omega in [-u_limit, u_limit], lambda in [0, inf]
    m_con = G_qp.shape[0]
    bupper = np.concatenate([
        np.full(N, u_limit),
        np.full(M, np.inf),
        h_qp,
    ])
    blower = np.concatenate([
        np.full(N, -u_limit),
        np.zeros(M),
        np.full(m_con, -np.inf),
    ])
    sense = np.zeros(n_var + m_con, dtype=np.int32)

    x, fval, flag, info = daqp.solve(
        H, f, G_qp, bupper, blower, sense,
        primal_tol=float(qp_eps_abs), dual_tol=float(qp_eps_abs),
        iter_limit=int(qp_max_iter), time_limit=float(qp_time_limit),
    )

    _last_daqp_solve_time = float(info.get('solve_time', 0.0))
    if flag == 1 or flag == 2:
        return x[:N].reshape(-1, 1), 'ok'
    reason = {-1: 'qp_infeasible', -2: 'qp_infeasible', -4: 'qp_iter_limit', -7: 'qp_time_limit'}.get(flag, f'qp_fail_{flag}')
    print(f"QP fail (daqp, flag={flag}, reason={reason}) at t={t}")
    return None, reason


In [ ]:
def calculate_std_dev(states_matrix, n_agents):
    """
    Spatial standard deviation of a state matrix (position or velocity).
    """
    if n_agents == 0:
        return 0.0
    mean_vec = np.mean(states_matrix, axis=1, keepdims=True)
    centered_matrix = states_matrix - mean_vec
    sum_of_squared_distances = np.sum(centered_matrix**2)
    std_dev = np.sqrt(sum_of_squared_distances / n_agents)
    return std_dev

In [ ]:
# ==============================================================================
# 2. Simulation loop (MA-CBF-VO version)
# ==============================================================================
# Side-channel for per-call failure diagnostics.
run_status = {'reason': None, 'time': None}

def run_multi_agent_simulation(N_AGENTS, V_CONST, critical_distance, initial_states_matrix):
    feasible = 1
    run_status['reason'] = 'ok'
    run_status['time'] = None

    position_std_dev = calculate_std_dev(initial_states_matrix[0:2, :], N_AGENTS)
    velocity_std_dev = calculate_std_dev(initial_states_matrix[2:4, :], N_AGENTS)

    T_FINAL = 600
    DT_CONTROL = 0.05

    u_limit = 0.35

    beta = np.load('parameter/beta.npy')
    lamda = np.load('parameter/lamda.npy')
    k1 = np.load('parameter/k1.npy')
    k2 = np.load('parameter/k2.npy')
    desired_distance = np.load(f'parameter/{CASE_ID}/desired_distance.npy')

    class_k1 = np.load('parameter/class_k1.npy')
    class_k2 = np.load('parameter/class_k2.npy')
    alpha_vo = float(class_k1)
    k_vo = float(np.load('parameter/k_vo.npy'))
    margin = float(np.load('parameter/margin.npy'))
    qp_max_iter = int(np.load('parameter/qp_max_iter.npy'))
    qp_eps_abs = float(np.load('parameter/qp_eps_abs.npy'))
    qp_time_limit = float(np.load('parameter/qp_time_limit.npy'))
    fi_threshold = float(np.load('parameter/fi_threshold.npy'))   # forward-invariance failure threshold (centralized)

    N = N_AGENTS
    M = (N_AGENTS * (N_AGENTS - 1)) // 2
    n_var = N + M
    n_eq  = 0
    n_in  = M + M + M + N

    y0 = initial_states_matrix.flatten('F')

    times = np.arange(0, T_FINAL, DT_CONTROL)
    history = [initial_states_matrix]
    control_history = []
    minimum_distance_history = []
    minimum_h_history = []
    global _qp_time_history_buf, _qp_daqp_history_buf, _u_nominal_history_buf
    _qp_time_history_buf = []
    _qp_daqp_history_buf = []
    _u_nominal_history_buf = []
    maximum_distance_history = []
    current_y = y0

    rows, cols = np.triu_indices(N_AGENTS, k=1)

    # --- NaN/Inf counter ---
    nan_inf_count = 0
    u_violation_count = 0

    print("5D Simulation starting (MA-CBF-VO)...")

    for t in times[:-1]:

        if t > 0 and t % 50 < DT_CONTROL:
            print(f"  Progress: t={t:.0f}/{T_FINAL}s ({t/T_FINAL*100:.1f}%)")

        current_states_matrix = current_y.reshape((STATE_DIM, N_AGENTS), order='F')
        current_states_matrix[4, :] = (current_states_matrix[4, :] + np.pi) % (2 * np.pi) - np.pi

        pij_norm, vij_norm, pv_norm, seta_ij, p_dot_seta = relative_state(
            STATE_DIM, N_AGENTS, current_states_matrix
        )

        minimum_inter_distance = minimum_inter_distance_function(pij_norm, N_AGENTS)
        minimum_distance_history.append(minimum_inter_distance)

        # Collision detection
        if minimum_inter_distance < critical_distance:
            run_status['reason'] = 'collision'; run_status['time'] = float(t)
            print(f"COLLISION at t={t:.3f}: min_dist={minimum_inter_distance:.4f} < critical_distance={float(critical_distance):.4f}")
            print(f"  [NaN/Inf occurrences: {nan_inf_count}]")
            feasible = 0
            return None, None, None, None, None, None, feasible, None, None, None, None

        maximum_distance_history.append(maximum_inter_distance_function(pij_norm, N_AGENTS))

        u_nominal = ACS_flocking(u_limit, beta, lamda, k1, k2, desired_distance,
                                  N_AGENTS, V_CONST, pij_norm, vij_norm, pv_norm,
                                  seta_ij, p_dot_seta)

        h1_vector, psi1_vector, ho_constraint, lgh_ho = ho_cbf(
            pij_norm, vij_norm, pv_norm, N_AGENTS, V_CONST,
            critical_distance, current_states_matrix, class_k1, class_k2, margin
        )

        h_vo_vector, lfh_vo, lgh_vo = vo_cbf(
            pij_norm, vij_norm, pv_norm, N_AGENTS, V_CONST,
            critical_distance, current_states_matrix, margin
        )

        # --- NaN/Inf check (count only, no termination) ---
        if np.any(np.isnan(ho_constraint)) or np.any(np.isinf(ho_constraint)) or \
           np.any(np.isnan(h_vo_vector)) or np.any(np.isinf(h_vo_vector)) or \
           np.any(np.isnan(lfh_vo)) or np.any(np.isinf(lfh_vo)) or \
           np.any(np.isnan(lgh_vo)) or np.any(np.isinf(lgh_vo)):
            nan_inf_count += 1

        pij_vectors = current_states_matrix[0:2, rows] - current_states_matrix[0:2, cols]
        vij_vectors = current_states_matrix[2:4, rows] - current_states_matrix[2:4, cols]
        T_col = time_to_collision_pairs(pij_vectors, vij_vectors, critical_distance)

        minimum_h = float(min(np.min(h1_vector), np.min(psi1_vector)))   # fi_fail if h1 OR psi_1 violates
        minimum_h_history.append(minimum_h)

        # --- Forward invariance check: is h1(x) below tolerance? ---
        if minimum_h < fi_threshold:
            run_status['reason'] = 'forward_invariance_fail'; run_status['time'] = float(t)
            print(f"FORWARD_INVARIANCE_FAIL at t={t:.3f}: min(h1, psi_1)={minimum_h:.3e} < {fi_threshold:.3e}")
            print(f"  [NaN/Inf occurrences: {nan_inf_count}]")
            feasible = 0
            return None, None, None, None, None, None, feasible, None, None, None, None

        _qp_t0 = time.perf_counter()
        optimal_u, qp_reason = solve_ma_cbf_vo_qp_with_warm_start(
            N, M, n_var, n_in,
            ho_constraint, lgh_ho,
            h_vo_vector, lfh_vo, lgh_vo,
            T_col, k_vo, alpha_vo,
            u_nominal, u_limit, t,
            qp_max_iter, qp_eps_abs, qp_time_limit
        )
        _qp_time_history_buf.append(time.perf_counter() - _qp_t0)
        _qp_daqp_history_buf.append(_last_daqp_solve_time)
        _u_nominal_history_buf.append(u_nominal.copy())

        if optimal_u is None:
            run_status['reason'] = qp_reason; run_status['time'] = float(t)
            print(f"infeasibility occur: {qp_reason}")
            print(f"  [NaN/Inf occurrences: {nan_inf_count}]")
            feasible = 0
            return None, None, None, None, None, None, feasible, None, None, None, None

        # --- u_limit violation check (diagnostic) ---
        max_abs_u = float(np.max(np.abs(optimal_u)))
        if max_abs_u > u_limit + qp_eps_abs:
            u_violation_count += 1
            if u_violation_count <= 5:
                print(f"U_LIMIT_VIOLATION at t={t:.3f}: |omega|_max={max_abs_u:.4f} > u_limit={u_limit:.4f}")

        control_history.append(optimal_u.copy())

        sol = solve_ivp(
            fun=multi_agent_const_speed_5d_dynamics,
            t_span=[t, t + DT_CONTROL],
            y0=current_y,
            args=(optimal_u, N_AGENTS, STATE_DIM, V_CONST)
        )
        current_y = sol.y[:, -1]
        history.append(current_y.reshape((STATE_DIM, N_AGENTS), order='F'))

    print("Simulation finished.")
    print("feasible:", feasible)
    print(f"  [NaN/Inf occurrences: {nan_inf_count}]")
    print(f"  [u_limit violations: {u_violation_count}]")

    final_states_matrix = history[-1]
    final_position_std_dev = calculate_std_dev(final_states_matrix[0:2, :], N_AGENTS)
    final_velocity_std_dev = calculate_std_dev(final_states_matrix[2:4, :], N_AGENTS)

    return (times, np.array(history), np.array(control_history),
            np.array(minimum_distance_history), np.array(maximum_distance_history),
            np.array(minimum_h_history), feasible, position_std_dev, velocity_std_dev,
            final_position_std_dev, final_velocity_std_dev)


In [ ]:
# --- Initial state setup (5-dimensional) ---

N_AGENTS = np.load(f'parameter/{CASE_ID}/N_AGENTS.npy')
V_CONST = np.load('parameter/V_CONST.npy')
critical_distance = np.load(f'parameter/{CASE_ID}/critical_distance.npy')
STATE_DIM = np.load('parameter/STATE_DIM.npy')
initial_test_case = np.load(f'initial_conditions/{CASE_ID}/initial.npy')
test_case_num = int(initial_test_case.shape[-1])   # derived from initial.npy shape

In [ ]:
ma_total_history = []
ma_total_control_history = []
ma_total_minimum_distance_history = []
ma_total_maximum_distance_history = []
ma_total_minimum_h_history = []
ma_total_qp_time_list = []
ma_total_daqp_solve_time_list = []
ma_total_u_nominal_history = []
ma_feasibility_list = []
ma_initial_position_std_dev_list = []
ma_initial_velocity_std_dev_list = []
ma_final_position_std_dev_list = []
ma_final_velocity_std_dev_list = []
ma_total_times_list = []

In [ ]:
# Run simulation
ma_failure_reason_list = []
ma_failure_time_list = []
for test_case in range(test_case_num):
    print(f"Running simulation for test case {test_case}...")
    initial_states_matrix = initial_test_case[:, :, test_case]
    (times, history, heading_angel_rate, smallist_distance, largiest_distance,
     smallist_h, feasiblity, initial_position_std_dev, initial_velocity_std_dev,
     final_position_std_dev, final_velocity_std_dev) = run_multi_agent_simulation(
        N_AGENTS, V_CONST, critical_distance, initial_states_matrix
    )
    ma_failure_reason_list.append(run_status['reason'])
    ma_failure_time_list.append(run_status['time'])

    if feasiblity == 1:
        print(f"Test case {test_case} is feasible and recorded.")
        ma_total_history.append(history)
        ma_total_control_history.append(heading_angel_rate)
        ma_total_minimum_distance_history.append(smallist_distance)
        ma_total_maximum_distance_history.append(largiest_distance)
        ma_total_minimum_h_history.append(smallist_h)
        ma_total_qp_time_list.append(list(_qp_time_history_buf))
        ma_total_daqp_solve_time_list.append(list(_qp_daqp_history_buf))
        ma_total_u_nominal_history.append(np.array(_u_nominal_history_buf))
        ma_feasibility_list.append(True)
        ma_initial_position_std_dev_list.append(initial_position_std_dev)
        ma_initial_velocity_std_dev_list.append(initial_velocity_std_dev)
        ma_final_position_std_dev_list.append(final_position_std_dev)
        ma_final_velocity_std_dev_list.append(final_velocity_std_dev)
        ma_total_times_list.append(times)
    else:
        print(f"Test case {test_case} is infeasible.")
        ma_total_history.append(None)
        ma_total_control_history.append(None)
        ma_total_minimum_distance_history.append(None)
        ma_total_maximum_distance_history.append(None)
        ma_total_minimum_h_history.append(None)
        ma_total_qp_time_list.append(None)
        ma_total_daqp_solve_time_list.append(None)
        ma_total_u_nominal_history.append(None)
        ma_feasibility_list.append(False)
        ma_initial_position_std_dev_list.append(None)
        ma_initial_velocity_std_dev_list.append(None)
        ma_final_position_std_dev_list.append(None)
        ma_final_velocity_std_dev_list.append(None)
        ma_total_times_list.append(None)

    feasible_count = sum(ma_feasibility_list)
    print(f"\n--- Simulation Summary ---")
    print(f"Feasible cases: {feasible_count} / {test_case_num}")
    if test_case_num > 0:
        feasibility_rate = (feasible_count / test_case_num) * 100
        print(f"Feasibility Rate: {feasibility_rate:.2f}%")

In [ ]:
ma_data = {
    'ma_total_history': ma_total_history,
    'ma_total_control_history': ma_total_control_history,
    'ma_total_minimum_distance_history': ma_total_minimum_distance_history,
    'ma_total_maximum_distance_history': ma_total_maximum_distance_history,
    'ma_total_minimum_h_history': ma_total_minimum_h_history,
    'ma_total_qp_time_list':ma_total_qp_time_list,
    'ma_total_daqp_solve_time_list':ma_total_daqp_solve_time_list,
    'ma_total_u_nominal_history':ma_total_u_nominal_history,
    'ma_feasibility_list': ma_feasibility_list,
    'ma_initial_position_std_dev_list': ma_initial_position_std_dev_list,
    'ma_initial_velocity_std_dev_list': ma_initial_velocity_std_dev_list,
    'ma_final_position_std_dev_list': ma_final_position_std_dev_list,
    'ma_final_velocity_std_dev_list': ma_final_velocity_std_dev_list,
    'ma_failure_reason_list': ma_failure_reason_list,
    'ma_total_times_list': ma_total_times_list,
}

with open(f"result/{CASE_ID}/ma/ma_simulation_data.pkl", "wb") as f:
    pickle.dump(ma_data, f)
    print("Saved! You should see ma_simulation_data.pkl in the folder.")